# Run views and queries

## 1.0 Import necessary libraries

In [1]:
# Connect to database
import pyodbc

# Data manipulation and analysis
import pandas as pd

# Path to folders and files
from pathlib import Path

# Path setup

In [2]:
# Current directory
CURRENT_DIR = Path.cwd()

# Path to main folder
PROJECT_ROOT = CURRENT_DIR.parent

# Path to folders and files
RESULTS_FOLDER = PROJECT_ROOT/"SQL_results"

In [3]:
# Configuration
conn_str = r'DRIVER={ODBC Driver 17 for SQL Server};SERVER=KIT\MSSQLSERVER01;DATABASE=RetailDB; Trusted_Connection=yes;'

conn = pyodbc.connect(conn_str)
print("Connection successful!")
#print('Connected to database succesfully')

# Create Views
cursor = conn.cursor()

# Customer view
cursor.execute("""
IF OBJECT_ID('customer_summary') iS NOT NULL DROP VIEW customer_summary
""")
cursor.execute("""
CREATE VIEW customer_summary AS
SELECT
    CustomerID,
    Country,
    COUNT(DISTINCT InvoiceNo) AS TotalOrders,
    SUM(TotalPrice) AS TotalSales,
    MIN(InvoiceDate) AS FirstPurchase,
    MAX(InvoiceDate) As LastPurchase,
    ROW_NUMBER() OVER (ORDER BY SUM(TotalPrice) DESC) AS SalesRank,
    ROW_NUMBER() OVER (ORDER BY COUNT(DISTINCT InvoiceNo) DESC) AS OrdersRank,
    ROW_NUMBER() OVER (PARTITION BY Country ORDER BY COUNT(DISTINCT InvoiceNo) DESC) AS CountryOrdersRank,
    ROW_NUMBER() OVER (PARTITION BY Country ORDER BY SUM(TotalPrice) DESC) AS CountrySalesRank
FROM online_retail
GROUP BY CustomerID, Country;
""")
print("Customer view created")

# Time view
cursor.execute("""
IF OBJECT_ID('time_summary') iS NOT NULL DROP VIEW time_summary
""")
cursor.execute("""
CREATE VIEW time_summary AS 
SELECT
    CAST(InvoiceDate AS DATE) AS Date,
    DATENAME(weekday, InvoiceDate) AS Weekday,
    DATEPART(weekday, InvoiceDate) AS WeekdayNum,
    DATEPART(hour, InvoiceDate) AS Hour,
    COUNT(DISTINCT InvoiceNo) AS OrderCount,
    SUM(TotalPrice) AS TotalSales
FROM online_retail
GROUP BY CAST(InvoiceDate AS DATE), DATENAME(weekday, InvoiceDate), DATEPART(weekday, InvoiceDate), DATEPART(hour, InvoiceDate);
""")
print("Time view created")

# Order view
cursor.execute("""
IF OBJECT_ID('order_summary') iS NOT NULL DROP VIEW order_summary
""")
cursor.execute("""
CREATE VIEW order_summary AS
SELECT
    FORMAT(InvoiceDate, 'yyyy-MM') AS OrderMonth,
    CustomerID,
    InvoiceNo,
    COUNT(DISTINCT InvoiceNo) As OrderCount,
    COUNT(DISTINCT StockCode) AS TotalDistinctProducts,
    SUM(Quantity) AS TotalQuantity,
    SUM(TotalPrice) AS TotalSales,
    Country,
    ROW_NUMBER() OVER (ORDER BY SUM(TotalPrice) DESC) AS SalesRank,
    ROW_NUMBER() OVER (PARTITION BY Country ORDER BY COUNT(DISTINCT InvoiceNo) DESC) AS CountryRank
FROM online_retail
GROUP BY CustomerID, InvoiceNo, Country, FORMAT(InvoiceDate, 'yyyy-MM');             
""")

print("Order view created")

# Products view
cursor.execute("""
IF OBJECT_ID('product_summary') iS NOT NULL DROP VIEW product_summary
""")
cursor.execute("""
CREATE VIEW product_summary AS
SELECT 
    StockCode,
    COUNT(DISTINCT StockCode) AS ProductCount,
    Description,
    Country,
    FORMAT(InvoiceDate, 'yyyy-MM') AS OrderMonth,
    SUM(Quantity) AS TotalQuantity,
    SUM(TotalPrice) AS TotalSales,
    ROW_NUMBER() OVER (ORDER BY SUM(TotalPrice) DESC) AS SalesRank,
    ROW_NUMBER() OVER (ORDER BY SUM(Quantity) DESC) AS QuantityRank,
    ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(TotalPrice) DESC) AS CountrySalesRank,
    ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(Quantity)DESC) AS CountryQuantityRank,
    ROW_NUMBER() OVER(PARTITION BY FORMAT(InvoiceDate, 'yyyy-MM') ORDER BY SUM(TotalPrice) DESC) AS MonthlySalesRank,
    ROW_NUMBER() OVER(PARTITION BY FORMAT(InvoiceDate, 'yyyy-MM') ORDER BY SUM(Quantity)DESC) AS MonthlyQuantityRank,
    ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(TotalPrice) ASC) AS CountrySalesRankAsc,
    ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(Quantity)ASC) AS CountryQuantityRankAsc,
    ROW_NUMBER() OVER(PARTITION BY FORMAT(InvoiceDate, 'yyyy-MM') ORDER BY SUM(TotalPrice) ASC) AS MonthlySalesRankAsc,
    ROW_NUMBER() OVER(PARTITION BY FORMAT(InvoiceDate, 'yyyy-MM') ORDER BY SUM(Quantity) ASC) AS MonthlyQuantityRankAsc
FROM online_retail
GROUP BY Country, StockCode, Description, FORMAT(InvoiceDate, 'yyyy-MM');
""")

print("product view created")

# Revenue view
cursor.execute("""
IF OBJECT_ID('revenue_summary') iS NOT NULL DROP VIEW revenue_summary
""")
cursor.execute("""
CREATE VIEW revenue_summary AS
SELECT
    Country,
    FORMAT(InvoiceDate, 'yyyy-MM') AS OrderMonth,
    SUM(TotalPrice) AS TotalSales,
    ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(TotalPrice) DESC) AS CountrySalesRank,
    ROW_NUMBER() OVER(PARTITION BY FORMAT(InvoiceDate, 'yyyy-MM') ORDER BY SUM(TotalPrice) DESC) AS MonthlySalesRank,
    ROW_NUMBER() OVER(PARTITION BY FORMAT(InvoiceDate, 'yyyy-MM') ORDER BY SUM(TotalPrice) ASC) AS MonthlySalesRankAsc,
    ROW_NUMBER() OVER(PARTITION BY Country ORDER BY SUM(TotalPrice) ASC) AS CountrySalesRankAsc
FROM online_retail
GROUP BY Country, FORMAT(InvoiceDate, 'yyyy-MM');               
""")

print("Revenue view created")

# rfm view
cursor.execute("""
IF OBJECT_ID('customer_rfm') iS NOT NULL DROP VIEW customer_rfm
""")
cursor.execute("""
CREATE VIEW customer_rfm AS
WITH last_date AS (
    SELECT MAX(InvoiceDate) AS CurrentDate
    FROM online_retail
),
rfm_calculations AS (
    -- Base metrics for each customer
    SELECT
        CustomerID,
        -- Recency: Days since the last purchase (how recently a customer has made a purchase)
        DATEDIFF(day, MAX(InvoiceDate), l.CurrentDate) AS Recency,
        -- Frequency: Count of invoices (how often a customer makes a purchase)
        COUNT(DISTINCT InvoiceNo) AS Frequency,
        -- Monetary: Total money spent (how much money a customer has spent over time)
        SUM(TotalPrice) AS Monetary
    FROM online_retail
    CROSS JOIN last_date l
    WHERE CustomerID IS NOT NULL
    GROUP BY CustomerID, l.CurrentDate
),
rfm_scoring AS (
    -- Convert metrics to 1-3 scores
    SELECT *,
        CASE
            WHEN Recency <=30 THEN 3
            WHEN Recency <=90 THEN 2
            ELSE 1
        END AS RecencyScore,
        CASE
            WHEN Frequency >= 60 THEN 3
            WHEN Frequency >=20 THEN 2
            ELSE 1
        END AS FrequencyScore,
        CASE
            WHEN Monetary >= 70000 THEN 3
            WHEN Monetary >= 9000 THEN 2
            ELSE 1
        END AS MonetaryScore
    FROM rfm_calculations
)
-- Assign customer segments based on combined scores
SELECT
    CustomerID,
    Recency,
    Frequency,
    Monetary,
    RecencyScore,
    FrequencyScore,
    MonetaryScore,
    CASE
        -- VIP : Highest tier 
        WHEN RecencyScore = 3 AND FrequencySCore = 3 AND MonetaryScore = 3 THEN 'VIP'
        
        -- Loyal Customers
        WHEN RecencyScore = 3 AND FrequencySCore >= 2 AND MonetaryScore >= 2 THEN 'Loyal Customers'
      
        -- Potential Loyalist
        WHEN RecencyScore = 3 AND FrequencySCore = 2 AND MonetaryScore = 2 THEN 'Potential Loyalist'
        
        -- New Customers
        WHEN RecencyScore = 3 AND FrequencySCore = 1 AND MonetaryScore <= 3 THEN 'New customers'
        
        -- Active
        WHEN RecencyScore >= 2 AND FrequencySCore >= 2 AND MonetaryScore >= 1 THEN 'Active'
        
        -- Need Attention
        WHEN RecencyScore >= 2 AND FrequencySCore <= 2 AND MonetaryScore <= 2 THEN 'Need Attention'

        -- Lost
        WHEN RecencyScore = 1 AND FrequencySCore = 1 AND MonetaryScore = 1 THEN 'Lost'
                   
        -- At risk
        WHEN RecencyScore = 1 AND FrequencySCore <= 3 AND MonetaryScore <= 3 THEN 'At Risk'
        
        ELSE 'Other'
    END AS CustomerSegment
FROM rfm_scoring;
""")

print("rfm view created")

# Recommendations view
cursor.execute("""
IF OBJECT_ID('recommendations') iS NOT NULL DROP VIEW recommendations
""")
cursor.execute("""
CREATE VIEW recommendations AS
    SELECT
        CustomerID,
        CustomerSegment,
        Recency,
        Frequency,
        Monetary,
        RecencyScore,
        FrequencyScore,
        MonetaryScore,
    --Actionable steps
    CASE
        WHEN CustomerSegment = 'VIP' THEN 'VIP access'  
        WHEN CustomerSegment = 'Loyal Customers' THEN 'Loyalty program with perks' 
        WHEN CustomerSegment = 'Potential Loyalist' THEN 'Rewards after every 10th order/ every other month + Personalised recommendation' 
        WHEN CustomerSegment = 'New Customers' THEN '15% Discount first order and second order' 
        WHEN CustomerSegment = 'Active' THEN '15% Discount voucher after every 10th/20th order + Personalised recommendation'
        WHEN CustomerSegment = 'Need Attention' THEN '15% Discount + free shipping + personalised emails/ chat messages/social media'
        WHEN CustomerSegment = 'At risk' THEN '20% Discount + Free shipping + personalised emails/chat messages/social media'
        WHEN CustomerSegment = 'Lost' THEN '25% Discount + emails/chat messages/social media'
        ELSE 'Standard Newsletter'
    END AS Recommendation
FROM customer_rfm;             
""")

print("recommendations view created")

conn.commit()
print("All views created")

# Dictionary of queries
queries = {
    # Customer analysis queries
    # Total number of customers 
    'total_customers': {
        'sql': """
            SELECT
                COUNT(DISTINCT CustomerID) AS CustomerCount
            FROM online_retail;
    """,
    'category': 'customer_analysis'
    },
    # Number of customers by country
    'customers_per_country': {
        'sql': """
            SELECT
                Country,
                COUNT(DISTINCT CustomerID) AS CustomerCount
            FROM online_retail
            GROUP BY Country
            ORDER BY CustomerCount DESC;
    """,
    'category': 'customer_analysis'
    },
    # Top 10 customers by total sales
    'top_10_customers_sales': {
        'sql': """
            SELECT TOP 10
                CustomerID,
                Country,
                TotalSales,
                DATEDIFF(day, CAST(FirstPurchase AS DATE), CAST(LastPurchase AS DATE)) AS CustomerLifetimeDays,
                OrdersRank
            FROM customer_summary
            ORDER BY TotalSales DESC;
    """,
    'category': 'customer_analysis'
    },
    # Bottom 10 customers by total sales
    'bottom_10_customers_sales': {
        'sql': """
            SELECT TOP 10
                CustomerID,
                Country,
                TotalSales,
                DATEDIFF(day, CAST(FirstPurchase AS DATE), CAST(LastPurchase AS DATE)) AS CustomerLifetimeDays,
                OrdersRank
            FROM customer_summary
            ORDER BY TotalSales ASC;
    """,
    'category': 'customer_analysis'
    },
    # Top 10 customers by number of orders
    'top_10_customers_orders': {
        'sql': """
            SELECT TOP 10
                CustomerID,
                Country,
                TotalOrders,
                DATEDIFF(day, CAST(FirstPurchase AS DATE), CAST(LastPurchase AS DATE)) AS CustomerLifetimeDays,
                SalesRank
            FROM customer_summary
            ORDER BY TotalOrders DESC;
    """,
    'category': 'customer_analysis'
    },
    # Bottom 10 customers by number of orders
    'bottom_10_customers_orders': {
        'sql': """
            SELECT TOP 10
                CustomerID,
                Country,
                TotalOrders,
                DATEDIFF(day, CAST(FirstPurchase AS DATE), CAST(LastPurchase AS DATE)) AS CustomerLifetimeDays,
                SalesRank
            FROM customer_summary
            ORDER BY TotalOrders ASC;
    """,
    'category': 'customer_analysis'
    },
    # Top customer in each country by total sales
    'top_customer_sales_per_country': {
        'sql': """
            SELECT 
                Country,
                CustomerID,
                DATEDIFF(day, CAST(FirstPurchase AS DATE), CAST(LastPurchase AS DATE)) AS CustomerLifetimeDays,
                TotalSales
            FROM customer_summary
            WHERE CountrySalesRank = 1
            ORDER BY TotalSales DESC;
    """,
    'category': 'customer_analysis'
    },
    # Top customer in each country by total orders
    'top_customer_orders_per_country': {
        'sql': """
            SELECT  
                Country,
                CustomerID,
                DATEDIFF(day, CAST(FirstPurchase AS DATE), CAST(LastPurchase AS DATE)) AS CustomerLifetimeDays,
                TotalOrders
            FROM customer_summary
            WHERE CountryOrdersRank = 1
            ORDER BY TotalOrders DESC;       
    """,
    'category': 'customer_analysis'
    },
  
    # Time analysis
    # Average daily revenue
    'average_daily_revenue': {
        'sql': """
            SELECT
                Weekday,
                ROUND(AVG(TotalSales), 1) AS AvgDailyRevenue
            FROM time_summary
            GROUP BY Weekday
            ORDER BY AvgDailyRevenue DESC;
    """,
    'category': 'time_analysis'
    },        
    # Average daily orders
        'average_daily_orders': {
        'sql': """
            SELECT
                Weekday,
                ROUND(AVG(OrderCount), 1) AS AvgDailyOrders
            FROM time_summary
            GROUP BY Weekday
            ORDER BY AvgDailyOrders DESC;
    """,
    'category': 'time_analysis'
    },
    # Top average hourly revenue
    'average_hourly_revenue': {
        'sql': """
            SELECT TOP 10
                Weekday,
                Hour,
                ROUND(AVG(TotalSales), 0) AS AvgHourlyRevenue,
                MAX(TotalSales) As PeakRevenue
            FROM time_summary
            GROUP BY Weekday, Hour
            ORDER BY AvgHourlyRevenue DESC;
    """,
    'category': 'time_analysis'
    },
    # Top average hourly orders
    'average_hourly_orders': {
        'sql': """
            SELECT TOP 10
                Weekday,
                Hour,
                ROUND(AVG(OrderCount), 0) AS AvgHourlyOrders,
                MAX(OrderCount) As PeakOrders
            FROM time_summary
            GROUP BY Weekday, Hour
            ORDER BY AvgHourlyOrders DESC;
    """,
    'category': 'time_analysis'
    },

    # Order Analysis
    # Total number of Orders
    'total_orders': {
            'sql': """
            SELECT 
                SUM(OrderCount) AS TotalOrders
            FROM order_summary;
    """,
    'category': 'order_analysis'
    },
    # Total Number of orders per month
    'orders_per_month': {
        'sql': """
            SELECT 
                OrderMonth,
                SUM(OrderCount) AS TotalOrders
            FROM order_summary
            GROUP BY OrderMonth
            ORDER BY OrderMonth DESC;
    """,
    'category': 'order_analysis'
    },
    # Total number of Orders per country
    'orders_per_country': {
        'sql': """
            SELECT 
                Country,
                SUM(OrderCount) AS TotalOrders   
            FROM order_summary
            GROUP BY Country
            ORDER BY TotalOrders DESC;
    """,
    'category': 'order_analysis'
    },
    # Top 5 months by number of orders
    'top_5_months_orders': {
        'sql': """
            SELECT TOP 5
                OrderMonth,
                SUM(OrderCount) AS TotalOrders
            FROM order_summary
            GROUP BY OrderMonth
            ORDER BY TotalOrders DESC;
    """,
    'category': 'order_analysis'
    },
    # Bottom 5 months with lowest number of orders
    'bottom_5_months_orders': {
        'sql': """
            SELECT TOP 5
                OrderMonth,
                SUM(OrderCount) AS TotalOrders 
            FROM order_summary
            GROUP BY OrderMonth
            ORDER BY TotalOrders ASC;
    """,
    'category': 'order_analysis'
    },
    # Number of items per order
    'total_items_per_order': {
        'sql': """
            SELECT  
                InvoiceNo,
                TotalDistinctProducts,
                TotalQuantity,
                TotalSales
            FROM order_summary
            ORDER BY TotalQuantity DESC;   
    """,
    'category': 'order_analysis'
    },

    # Product analysis
    # Total number of distinct products
    'total_distinct_products': {
        'sql': """
            SELECT
                SUM(ProductCount)
            FROM product_summary;
    """,
    'category': 'product_analysis'
    },
    # Top 10 products by total sales
    'top_10_products_sales': {
        'sql': """
            SELECT TOP 10
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalSales) AS ProductTotal
            FROM product_summary
            GROUP BY StockCode
            ORDER BY ProductTotal DESC;
    """,
    'category': 'product_analysis'
    },
    # Bottom 10 products by total sales
    'bottom_10_products_sales': {
        'sql': """
            SELECT TOP 10
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalSales) AS ProductTotal
            FROM product_summary
            GROUP BY StockCode
            ORDER BY ProductTotal ASC;
    """,
    'category': 'product_analysis'
    },
    # Top 10 products by Quantity sold
    'top_10_products_quantity': {
        'sql': """
            SELECT TOP 10
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalQuantity) AS ProductQuantity
            FROM product_summary
            GROUP BY StockCode
            ORDER BY ProductQuantity DESC;
    """,
    'category': 'product_analysis'
    },
    # Bottom 10 products by Quantity sold
    'bottom_10_products_quantity': {
        'sql': """
            SELECT TOP 10
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalQuantity) AS ProductQuantity
            FROM product_summary
            GROUP BY StockCode
            ORDER BY ProductQuantity ASC;
    """,
    'category': 'product_analysis'
    },
    # Top 3 products in each country by Sales
    'top_3_per_country_by_sales': {
        'sql': """
            SELECT 
                Country,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalSales) AS ProductTotal
            FROM product_summary
            WHERE CountrySalesRank <=3
            GROUP BY StockCode, Country
            ORDER BY Country, ProductTotal DESC;
    """,
    'category': 'product_analysis'
    },
    # Bottom 3 products in each country by Sales
    'bottom_3_per_country_by_sales': {
        'sql': """
            SELECT
                Country,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalSales) AS ProductTotal
            FROM product_summary
            WHERE CountrySalesRankAsc <=3
            GROUP BY StockCode, Country
            ORDER BY Country, ProductTotal;
    """,
    'category': 'product_analysis'
    },
    # Top 3 products in each country by quantity
    'top_3_per_country_by_quantity': {
        'sql': """
            SELECT 
                Country,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalQuantity) AS ProductQuantity
            FROM product_summary
            WHERE CountryQuantityRank <=3
            GROUP BY StockCode, Country 
            ORDER BY Country, ProductQuantity DESC;
    """,
    'category': 'product_analysis'
    },
    # Bottom 3 products in each country by quantity
    'bottom_3_per_country_by_quantity': {
        'sql': """
            SELECT 
                Country,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalQuantity) AS ProductQuantity
            FROM product_summary
            WHERE CountryQuantityRankAsc <=3
            GROUP BY StockCode, Country
            ORDER BY Country, ProductQuantity;
    """,
    'category': 'product_analysis'
    },
    # Top 3 products by Sales for each month
    'top_3_per_month_by_sales': {
        'sql': """
            SELECT 
                OrderMonth,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalSales) AS ProductTotal
            FROM product_summary
            WHERE MonthlySalesRank <=3
            GROUP BY StockCode, OrderMonth
            ORDER BY OrderMonth, ProductTotal DESC;
    """,
    'category': 'product_analysis'
    },
    # Bottom 3 by Sales for each month
    'bottom_3_per_month_by_sales': {
        'sql': """
            SELECT 
                OrderMonth,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalSales) AS ProductTotal
            FROM product_summary
            WHERE MonthlySalesRankAsc <=3
            GROUP BY StockCode, OrderMonth
            ORDER BY OrderMonth, ProductTotal;
    """,
    'category': 'product_analysis'
    },
    # Top 3 by Quantity for each month 
    'top_3_per_month_by_quantity': {
        'sql': """
            SELECT
                OrderMonth,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalQuantity) AS ProductQuantity
            FROM product_summary
            WHERE MonthlyQuantityRank <=3
            GROUP BY StockCode, OrderMonth
            ORDER BY OrderMonth, ProductQuantity DESC;
    """,
    'category': 'product_analysis'
    },
    # Bottom 3 by Quantity for each month 
    'bottom_3_per_month_by_quantity': {
        'sql': """
            SELECT
                OrderMonth,
                StockCode,
                MAX(Description) AS ProductDescription,
                SUM(TotalQuantity) AS ProductQuantity
            FROM product_summary
            WHERE MonthlySalesRankAsc <=3
            GROUP BY StockCode, OrderMonth
            ORDER BY OrderMonth, ProductQuantity;
    """,
    'category': 'product_analysis'
    },

    # Revenue Analysis
    # Total revenue
    'total_revenue': {
        'sql': """
            SELECT
                SUM(TotalSales) AS TotalRevenue
            FROM revenue_summary;
    """,
    'category': 'revenue_analysis'
    },
    # Total revenue per month
    'monthly_revenue': {
        'sql': """
            SELECT
                OrderMonth,
                SUM(TotalSales) AS TotalRevenue
            FROM revenue_summary
            GROUP BY OrderMonth
            ORDER BY OrderMonth, TotalRevenue;
    """,
    'category': 'revenue_analysis'
    },
    # Top 5 months with highest revenue
    'top_5_months_revenue': {
        'sql': """
            SELECT TOP 5    
                OrderMonth,
                SUM(TotalSales) AS TotalRevenue
            FROM revenue_summary
            GROUP BY OrderMonth
            ORDER BY TotalRevenue DESC;
    """,
    'category': 'revenue_analysis'
    },
    # Bottom 5 months revenue
    'bottom_5_months_revenue': {
        'sql': """
            SELECT TOP 5 
                OrderMonth,
                SUM(TotalSales) AS TotalRevenue
            FROM revenue_summary
            GROUP BY OrderMonth
            ORDER BY TotalRevenue ASC;
    """,
    'category': 'revenue_analysis'
    },
    # Revenue growth rate per month
    'revenue_growth_rate': {
        'sql': """
            SELECT
                OrderMonth,
                SUM(TotalSales) AS TotalRevenue,
                ROUND(
                    100 * (SUM(TotalSales) - LAG(SUM(TotalSales), 1) OVER (ORDER BY OrderMonth)) /
                    NULLIF(LAG(SUM(TotalSales), 1) OVER (ORDER BY OrderMonth), 0), 2 
                )AS GrowthRate
            FROM revenue_summary
            GROUP BY OrderMonth
            ORDER BY OrderMonth, TotalRevenue; 
    """,       
    'category': 'revenue_analysis'
    },  
    # Total revenue per country
    'revenue_per_country': {
        'sql': """
            SELECT
                Country,
                SUM(TotalSales) AS TotalRevenue
            FROM revenue_summary
            GROUP BY Country
            ORDER BY TotalRevenue DESC;
    """,
    'category': 'revenue_analysis'
    },
  
    # rfm analysis
    # Customer segments distribution
    'segments_distribution': {
        'sql': """
            SELECT
                CustomerSegment,
                COUNT(*) AS CustomerCount,
                ROUND(100.0 * COUNT(*)/ SUM(COUNT(*)) OVER(), 2) AS Percentage,
                AVG(Recency) AS AvgRecency,
                AVG(Frequency) AS AvgFrequency,
                AVG(Monetary) AS AvgMonetary
            FROM customer_rfm
            GROUP BY CustomerSegment
            ORDER BY AvgMonetary DESC;
    """,
    'category': 'rfm_analysis'
    },
    # Top 10 customers VIP
    'top_10_VIP': {
        'sql': """
            SELECT TOP 10
                CustomerID,
                CustomerSegment,
                Recency,
                Frequency,
                Monetary,
                RecencyScore,
                Frequencyscore,
                MonetaryScore
            FROM customer_rfm
            WHERE CustomerSegment = 'VIP'
            ORDER BY Monetary DESC;
    """,
    'category': 'rfm_analysis'
    },
    # Customers that are at risk/need more attention
    'top_20_at_risk': {
        'sql': """
            SELECT TOP 20
                CustomerID,
                 CustomerSegment,
                Recency,
                Frequency,
                Monetary,
                RecencyScore,
                Frequencyscore,
                MonetaryScore
            FROM customer_rfm
            WHERE CustomerSegment IN ('At Risk', 'Need Attention')
            ORDER BY Monetary DESC;
    """,
    'category': 'rfm_analysis'
    },
    # Revenue per segment
    'revenue_per_segment': {
        'sql': """
            SELECT
                CustomerSegment,
                SUM(Monetary) AS TotalRevenue,
                ROUND(100.0 * SUM(Monetary) / SUM(SUM(Monetary)) OVER(), 2) AS RevenuePercentage
            FROM customer_rfm
            GROUP BY CustomerSegment
            ORDER BY TotalRevenue DESC;
    """,
    'category': 'rfm_analysis'
    },
    # Revenue per segment
    'recommendation_per_segment': {
        'sql': """
            SELECT
                CustomerSegment,
                SUM(Monetary) AS TotalRevenue,
                ROUND(100.0 * SUM(Monetary) / SUM(SUM(Monetary)) OVER(), 2) AS RevenuePercentage,
                Recommendation
            FROM recommendations
            GROUP BY CustomerSegment, Recommendation;
    """,
    'category': 'segments_recommendation'
    }   
}

# Run each query and save results
for query_name, query_file in queries.items():
    category = query_file['category']
    output_folder = RESULTS_FOLDER / category
    output_folder.mkdir(parents=True, exist_ok=True)
    print(f"Processing: {query_name}")

    # Run query
    sql_df = pd.read_sql(query_file['sql'], conn)

    # Save results
    output_path =  output_folder/ f"{query_name}.csv"
    sql_df.to_csv(output_path, index=False, encoding='utf-8-sig')

    print(f" {query_name} - {len(sql_df)} rows saved.")
    

# close connection
conn.close()


Connection successful!
Customer view created
Time view created
Order view created
product view created
Revenue view created
rfm view created
recommendations view created
All views created
Processing: total_customers


C:\Users\Linda\AppData\Local\Temp\ipykernel_976\4287849434.py:827: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sql_df = pd.read_sql(query_file['sql'], conn)


 total_customers - 1 rows saved.
Processing: customers_per_country
 customers_per_country - 38 rows saved.
Processing: top_10_customers_sales
 top_10_customers_sales - 10 rows saved.
Processing: bottom_10_customers_sales
 bottom_10_customers_sales - 10 rows saved.
Processing: top_10_customers_orders
 top_10_customers_orders - 10 rows saved.
Processing: bottom_10_customers_orders
 bottom_10_customers_orders - 10 rows saved.
Processing: top_customer_sales_per_country
 top_customer_sales_per_country - 38 rows saved.
Processing: top_customer_orders_per_country
 top_customer_orders_per_country - 38 rows saved.
Processing: average_daily_revenue
 average_daily_revenue - 6 rows saved.
Processing: average_daily_orders
 average_daily_orders - 6 rows saved.
Processing: average_hourly_revenue
 average_hourly_revenue - 10 rows saved.
Processing: average_hourly_orders
 average_hourly_orders - 10 rows saved.
Processing: total_orders
 total_orders - 1 rows saved.
Processing: orders_per_month
 orders_p